# E-Commerce Review Sentiment Analysis (DistilBERT)

Production-style NLP pipeline for classifying product reviews into **negative** and **positive** sentiment.

## Why this project matters
- Demonstrates transfer learning with Hugging Face Transformers.
- Handles class imbalance using weighted loss.
- Includes reproducible training, structured evaluation, and model export for inference.

## Project workflow

1. Tokenizer
2. Token IDs + Attention Mask
3. Dataset Class
4. DistilBERT Model
5. Trainer API (training)
6. Predictions
7. Evaluation Metrics

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    Trainer,
    TrainingArguments,
)
import re
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import classification_report

from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

SEED = 42


d:\vscode\E-Commerce-Review-Intelligence-System\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Prefer a local data/ folder for portability.
DATA_PATH = Path("data/7817_1.csv")

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")

df.sample(5, random_state=SEED)

Dataset shape: (1597, 27)


,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
802,AVpe7LD5LJeJML43ybWA,"B00DOPNO4M,B00BWYQ9YE,B00CYQPMJC,B00CUU1CGY,B0...",Amazon,"Amazon Devices,Kindle Store,buy a kindle",NaN,2015-05-22T15:33:59Z,2017-07-18T23:52:40Z,NaN,NaN,"kindlefirehdx7/b00dopno4m,kindlefirehdx7/b00bw...",...,NaN,http://www.amazon.com/kindle-fire-hdx-student-...,I love this handheld device especially all the...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124,AVpfpzCi1cnluZ0-oxEr,B00DOPNLJ0,Amazon,Amazon Devices,NaN,2015-06-02T08:44:19Z,2017-08-07T15:41:59Z,NaN,NaN,kindlefirehdx89/b00dopnlj0,...,NaN,http://www.amazon.com/Kindle-Fire-HDX-Display-...,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,NaN,NaN,B. Tarbuck,NaN,NaN,NaN
350,AVpfLiCSilAPnD_xWpk_,B00CX5P8FC,Amazon,"Categories,Amazon Devices,Electronics Features...",NaN,2015-05-22T18:12:20Z,2017-08-08T22:03:26Z,NaN,8.487190e+11,"848719022827,0848719022827,amazonfiretv/b00cx5...",...,NaN,http://www.amazon.com/Fire-TV-streaming-media-...,This was easy to set up I can access many movi...,Amazon Fire TV - A must have!!!!,NaN,NaN,Amazon Customer,NaN,8.487190e+11,NaN
682,AVzRlo37glJLPUi8FbPy,B01LW1MS9C,Amazon,"Amazon Devices,Kindle Store",NaN,2017-06-22T20:55:23Z,2017-08-13T08:28:46Z,NaN,NaN,amazonechodotcasefitsechodot2ndgenerationonlyc...,...,5.0,https://www.amazon.com/Amazon-Echo-Case-fits-G...,I am thoroughly impressed with my Echo Dot and...,The Extra Touch,NaN,NaN,dm,NaN,NaN,NaN
1324,AVpfpK8KLJeJML43BCuD,B01BH83OOM,Amazon,"Amazon Devices,Home,Smart Home & Connected Liv...",Black,2017-01-04T03:51:17Z,2017-08-13T08:31:07Z,4.8 in x 6.6 in x 3.2 in,8.416670e+11,amazontapalexaenabledportablebluetoothspeaker/...,...,5.0,http://reviews.bestbuy.com/3545/5097300/review...,Great little device easy to connect to bluetoo...,Easy small decent sound,NaN,NaN,Drjim,NaN,8.416670e+11,1.75 lbs


## 1. Text Preprocessing and Label Engineering

In [3]:
df=df[["reviews.text", "reviews.rating"]]

- This are the only columns which are required for the sentiment analysis of Review

In [5]:
df.sample(5)

,reviews.text,reviews.rating
1533,While I've purchased items from Amazon for yea...,3.0
1304,i got to say that love it!! makes it easier to...,5.0
1458,While I've purchased items from Amazon for yea...,3.0
1173,Would buy the cheaper one than I bought.The ec...,3.0
882,Picked this up to have a portable option for p...,4.0


In [78]:
df.isnull().sum()

reviews.text        0
reviews.rating    420
dtype: int64

In [ ]:
# Keep only rows with explicit rating labels for supervised sentiment training.
df = df.dropna(subset=["reviews.rating"]).copy()

In [80]:
df.isnull().sum()

reviews.text      0
reviews.rating    0
dtype: int64

- Managed the Null values

In [ ]:
df = df.drop_duplicates().copy()
print(f"Samples after deduplication: {len(df)}")

,reviews.text,reviews.rating
0,I initially had trouble deciding between the p...,5.0
1,Allow me to preface this with a little history...,5.0
2,I am enjoying it so far. Great for reading. Ha...,4.0
3,I bought one of the first Paperwhites and have...,5.0
4,I have to say upfront - I don't like coroporat...,5.0
...,...,...
1592,This is not the same remote that I got for my ...,3.0
1593,I have had to change the batteries in this rem...,1.0
1594,"Remote did not activate, nor did it connect to...",1.0
1595,It does the job but is super over priced. I fe...,3.0


In [27]:
# Keep only clearly polar reviews for cleaner sentiment supervision.
df = df[df["reviews.rating"].isin([1, 2, 4, 5])].copy()

def label_sentiment(rating: float) -> int:
    return 1 if rating >= 4 else 0

df["label"] = df["reviews.rating"].apply(label_sentiment)

print("Class distribution:")
print(df["label"].value_counts(normalize=True).rename("ratio"))
print(f"Total samples after dropping 3-star reviews: {len(df)}")

Class distribution:
label
1    0.927825
0    0.072175
Name: ratio, dtype: float64
Total samples after dropping 3-star reviews: 1053


- Here i have crated a binary label_sentiment for the reviews positive (1) fir >=4 and negative(0) otherwise
- Here we can see that there is class imbalance which further will manage it

In [28]:
df.sample(5, random_state=SEED)

,reviews.text,reviews.rating,label
756,"whether you just want the weather, latest spor...",5.0,1
200,i am writing this from the perspective of bein...,4.0,1
1411,"so far, this is doing what i wanted it to. i p...",5.0,1
1596,i ordered this item to replace the one that no...,1.0,0
159,i am updating my review from two months ago. m...,4.0,1


In [29]:
import re
import unicodedata
import pandas as pd

# Compile once for speed
URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
HTML_RE = re.compile(r"<[^>]+>")
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
MULTI_SPACE_RE = re.compile(r"\s+")
REPEAT_PUNCT_RE = re.compile(r"([!?.,;:])\1{1,}")   # !!! -> !
# Keep letters, numbers, whitespace, and punctuation useful for clause/aspect boundaries
ALLOWED_RE = re.compile(r"[^a-z0-9\s\.,!?;:]")

CONTRACTION_RULES = [
    (re.compile(r"\bwon't\b"), "will not"),
    (re.compile(r"\bcan't\b"), "can not"),
    (re.compile(r"\bshan't\b"), "shall not"),
    (re.compile(r"n't\b"), " not"),
    (re.compile(r"'re\b"), " are"),
    (re.compile(r"'ve\b"), " have"),
    (re.compile(r"'ll\b"), " will"),
    (re.compile(r"'d\b"), " would"),
    (re.compile(r"'m\b"), " am"),
    (re.compile(r"'s\b"), " is")
]

def clean_review_text(text: str) -> str:
    # null-safe
    text = "" if pd.isna(text) else str(text)

    # normalize unicode and apostrophes
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("’", "'").replace("`", "'")

    # lowercase for distilbert-base-uncased
    text = text.lower()

    # remove obvious noise
    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)

    # expand contractions BEFORE symbol filtering
    for pattern, replacement in CONTRACTION_RULES:
        text = pattern.sub(replacement, text)

    # light cleanup only
    text = ALLOWED_RE.sub(" ", text)
    text = REPEAT_PUNCT_RE.sub(r"\1", text)
    text = MULTI_SPACE_RE.sub(" ", text).strip()

    return text

df["reviews.text"] = df["reviews.text"].apply(clean_review_text)
df = df[df["reviews.text"].str.len() > 0].copy()
df[["reviews.text"]].head()

,reviews.text
0,i initially had trouble deciding between the p...
1,allow me to preface this with a little history...
2,i am enjoying it so far. great for reading. ha...
3,i bought one of the first paperwhites and have...
4,i have to say upfront i do not like coroporate...


- Preserves negation meaning by expanding contractions like do not, can not.
- Keeps useful punctuation for mixed/aspect sentiment clues.
- Removes noisy artifacts only, so transformer context is preserved.

Here the using the stopwords and lemmatization will loose some context like the negations  skippinfg lemmatization here is better since It adds mismatch risk if inference text is not lemmatized exactly the same way.
and the DistilBerttokenizer transformers learn the word forms in context whcih is better

### Train Test Split

In [30]:

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["reviews.text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

- Here tolist() is important for the Pytorch Implementation beacause of which it will further cause error in the tokenizing step

## 2. Tokenization
- Converts text → numbers (tokens)
- max_length → limit tokens
- Output of tokenization will be 
input_ids → token numbers
attention_mask → tells model which tokens are real vs padding

In [31]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=32
)
# The DistilBert supports maximum lenght upto 512, here 128 is a good starting point
test_encodings = tokenizer(
    test_texts,
    truncation=True, #Cuts off reviews that are longer than max_length tokens.
    padding=True,
    max_length=32
)

###  Create PyTorch Datasets

- PyTorch requires data in a Dataset format

 ___ getitem ___() For each index:

self.encodings is a dictionary from the tokenizer, usually with keys like input_ids and attention_mask.
For each key, it takes the value at index idx: val[idx].
It wraps each value as a torch tensor.
Result is a dictionary called item containing one review’s tokenized inputs

item["labels"] = torch.tensor(self.labels[idx])
adds the ground-truth class for that sample.


input_ids: tensor(...)
attention_mask: tensor(...)
labels: tensor(1)
This is what model sees during training

In [32]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [33]:
train_dataset = ReviewDataset(train_encodings, train_labels)
test_dataset = ReviewDataset(test_encodings, test_labels)

### Compute Metrics Function

- It receives model outputs as eval_pred, then splits them into logits (raw prediction scores) and labels (true classes).
It converts logits to predicted classes using argmax, which picks the class with the highest score for each sample.

In [34]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    return {"accuracy": acc, "f1": f1}


### Load the Model

In [35]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
# here the senitment is just positive and negative so labels= 2

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1760.69it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Training Setup
### The HuggingFace Implementation 

- The TrianingArguments provides optimization

In [42]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### Handeling Class Imbalance 

### Compute Weights

- compute_class_weight(..., class_weight='balanced') calculates higher weight for the minority class and lower weight for the majority class.
- tells it which class IDs exist (for example 0 and 1).
- The result is converted to a PyTorch tensor with torch.tensor(..., dtype=torch.float) so it can be used in loss functions like CrossEntropyLoss(weight=...).

In [43]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

tensor([6.9016, 0.5391])


### Make a custom Trainer for manging the class imbalance

In [44]:
from torch.nn import CrossEntropyLoss
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **_kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

### Trainer
This cell creates the Hugging Face Trainer object, which is the engine that runs training and evaluation.

- Training loop
- Backpropagation
- Optimizer
- Evaluation loop

Hugging Face handles everything internally

In [45]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

### Train the Model

In [46]:
trainer.train()

d:\vscode\E-Commerce-Review-Intelligence-System\torch\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.388369,0.867299,0.924731
2,0.312255,0.386801,0.943128,0.969072
3,0.312255,0.501070,0.971564,0.984848
4,0.073449,0.644355,0.962085,0.979899


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.13s/it]
d:\vscode\E-Commerce-Review-Intelligence-System\torch\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.61it/s]
d:\vscode\E-Commerce-Review-Intelligence-System\torch\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.20it/s]
d:\vscode\E-Commerce-Review-Intelligence-System\torch\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|████

TrainOutput(global_step=108, training_loss=0.18040184759431416, metrics={'train_runtime': 314.5601, 'train_samples_per_second': 10.707, 'train_steps_per_second': 0.343, 'total_flos': 27884387417088.0, 'train_loss': 0.18040184759431416, 'epoch': 4.0})

### Prediction Function

torch.argmax(..., axis=1) picks the class with the highest score for each sample (0 or 1).



In [47]:
def get_predictions(dataset):
    outputs = trainer.predict(dataset)
    preds = torch.argmax(torch.tensor(outputs.predictions), axis=1) # picks the max among the each sample
    return preds

train_preds = get_predictions(train_dataset)
test_preds = get_predictions(test_dataset)


d:\vscode\E-Commerce-Review-Intelligence-System\torch\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


d:\vscode\E-Commerce-Review-Intelligence-System\torch\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


### Model Evaluation

In [48]:
print("\n🔹 TRAIN REPORT:")
print(classification_report(train_labels, train_preds))

print("\n🔹 TEST REPORT:")
print(classification_report(test_labels, test_preds))

print("\nTrain Accuracy:", accuracy_score(train_labels, train_preds))
print("Test Accuracy:", accuracy_score(test_labels, test_preds))

print("\nTrain F1:", f1_score(train_labels, train_preds))
print("Test F1:", f1_score(test_labels, test_preds))


🔹 TRAIN REPORT:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        61
           1       1.00      1.00      1.00       781

    accuracy                           1.00       842
   macro avg       0.99      0.99      0.99       842
weighted avg       1.00      1.00      1.00       842


🔹 TEST REPORT:
              precision    recall  f1-score   support

           0       0.91      0.67      0.77        15
           1       0.97      0.99      0.98       196

    accuracy                           0.97       211
   macro avg       0.94      0.83      0.88       211
weighted avg       0.97      0.97      0.97       211


Train Accuracy: 0.997624703087886
Test Accuracy: 0.9715639810426541

Train F1: 0.998719590268886
Test F1: 0.9848484848484849


## 5. Evaluation with Custom Examples

Tokenizer → converts to input_ids + attention_mask
   ↓
Model (DistilBERT)
   ↓
Logits (raw scores)
   ↓
Softmax → probabilities
   ↓
Argmax → final class (0 or 1)

In [49]:
def predict_sentiment(text):
    model.eval()
    cleaned_text = clean_review_text(text)
    
    inputs = tokenizer(
        cleaned_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=32
    )

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()

    id2label = model.config.id2label or {}
    pred_label = id2label.get(pred_id, id2label.get(str(pred_id), str(pred_id)))

    return {
        "text": text,
        "cleaned_text": cleaned_text,
        "prediction_id": pred_id,
        "prediction_label": pred_label,
        "confidence": probs[0][pred_id].item()
    }

In [50]:
print(predict_sentiment("This product is amazing, I loved it!"))
print(predict_sentiment("Worst experience ever, totally disappointed"))

{'text': 'This product is amazing, I loved it!', 'cleaned_text': 'this product is amazing, i loved it!', 'prediction_id': 1, 'prediction_label': 'LABEL_1', 'confidence': 0.9953957200050354}
{'text': 'Worst experience ever, totally disappointed', 'cleaned_text': 'worst experience ever, totally disappointed', 'prediction_id': 0, 'prediction_label': 'LABEL_0', 'confidence': 0.9893594980239868}


## 7. Save Model Artifacts

In [ ]:
import json

id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

# Persist label metadata in model config for safe downstream loading.
model.config.id2label = id2label
model.config.label2id = label2id

model.save_pretrained("sentiment_model")
tokenizer.save_pretrained("sentiment_model")

with open("sentiment_model/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "id2label": {str(k): v for k, v in id2label.items()},
            "label2id": label2id,
        },
        f,
        indent=2,
    )

eval_metrics = trainer.evaluate()
with open("sentiment_model/eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, indent=2)

preprocess_config = {
    "uses_clean_review_text": True,
    "contraction_expansion": True,
    "url_html_email_removal": True,
    "kept_punctuation": [".", ",", "!", "?", ";", ":"],
    "max_length": 32,
}
with open("sentiment_model/preprocess_config.json", "w", encoding="utf-8") as f:
    json.dump(preprocess_config, f, indent=2)

print("Saved model, tokenizer, label mapping, eval metrics, and preprocessing config.")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]


('sentiment_model\\tokenizer_config.json', 'sentiment_model\\tokenizer.json')